# ML Assignment 2 - Breast Cancer Classification (from scratch)

Five classifiers, the scaler and all metrics are implemented **from scratch with NumPy only** in `ml_models.py` (no scikit-learn): Logistic Regression (gradient descent), Decision Tree (Gini), KNN, Gaussian Naive Bayes, SVM (RBF kernel, simplified SMO). Target: `0` = Malignant, `1` = Benign.

This notebook trains them, evaluates them and saves `model/*.joblib` (same as `train_models.py`).

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from ml_models import (make_models, MODEL_FILES, confusion, accuracy, auc, precision, recall, f1, mcc)

## 1. Load data and train/test split
Test set = `test_data.csv` (20 %, stratified, 114 rows); training set = the other 455 rows.

In [2]:
full = pd.read_csv("breast_cancer_uci.csv")
test = pd.read_csv("test_data.csv")
features = [c for c in full.columns if c != "target"]
key = lambda d: d[features].round(8).astype(str).agg("|".join, axis=1)
train = full[~key(full).isin(set(key(test)))].reset_index(drop=True)

X_train, y_train = train[features].to_numpy(float), train["target"].to_numpy(int)
X_test, y_test = test[features].to_numpy(float), test["target"].to_numpy(int)
print(full.shape, X_train.shape, X_test.shape)
print("train class counts:", np.bincount(y_train), "test class counts:", np.bincount(y_test))

(569, 31) (455, 30) (114, 30)
train class counts: [170 285] test class counts: [42 72]


## 2. Train all five models and evaluate on the test set

In [3]:
models = make_models()
results = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred, y_prob = pipe.predict(X_test), pipe.predict_proba(X_test)
    results.append({
        "ML Model Name": name,
        "Accuracy": round(accuracy(y_test, y_pred), 4),
        "AUC": round(auc(y_test, y_prob), 4),
        "Precision": round(precision(y_test, y_pred), 4),
        "Recall": round(recall(y_test, y_pred), 4),
        "F1": round(f1(y_test, y_pred), 4),
        "MCC": round(mcc(y_test, y_pred), 4),
    })
pd.DataFrame(results)

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9737,0.9960,0.9859,0.9722,0.9790,0.9439
1,Decision Tree,0.9211,0.9163,0.9565,0.9167,0.9362,0.8341
2,KNN,0.9737,0.9884,0.9600,1.0000,0.9796,0.9442
3,Naive Bayes,0.9298,0.9868,0.9444,0.9444,0.9444,0.8492
4,Support Vector Machine,0.9825,0.9950,0.9861,0.9861,0.9861,0.9623


## 3. Confusion matrices

In [4]:
for name, pipe in models.items():
    tp, tn, fp, fn = confusion(y_test, pipe.predict(X_test))
    print(f"{name:25s} TN={tn:3d} FP={fp:3d} FN={fn:3d} TP={tp:3d}")

Logistic Regression       TN= 41 FP=  1 FN=  2 TP= 70
Decision Tree             TN= 39 FP=  3 FN=  6 TP= 66
KNN                       TN= 39 FP=  3 FN=  0 TP= 72
Naive Bayes               TN= 38 FP=  4 FN=  4 TP= 68
Support Vector Machine    TN= 41 FP=  1 FN=  1 TP= 71


## 4. Save each trained model as .joblib
Each file is a scaler + model pipeline that accepts raw features; `app.py` only loads it and calls `.predict()` / `.predict_proba()`.

In [5]:
os.makedirs("model", exist_ok=True)
for name, pipe in models.items():
    joblib.dump(pipe, f"model/{MODEL_FILES[name]}.joblib")
    print("saved", f"model/{MODEL_FILES[name]}.joblib")

saved model/logistic_regression.joblib
saved model/decision_tree.joblib
saved model/knn.joblib
saved model/naive_bayes.joblib
saved model/support_vector_machine.joblib
